In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc


yellow_keep = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount"
]

# test cleaning for one year first - 2024
# inspect jan 2024 first 
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

df = pd.read_parquet(url, columns=yellow_keep)

print("Working shape:", df.shape)
print("Approx memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))
display(df.head())

# 1.initial inspection
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)
display(missing_summary)

display(df["passenger_count"].value_counts(dropna=False).sort_index().to_frame("count"))
display(df["trip_distance"].describe().to_frame("trip_distance"))
display(df["fare_amount"].describe().to_frame("fare_amount"))


# 2.type screening
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

display(pd.DataFrame(df.dtypes, columns=["dtype"]))


# 3.minimal early cleaning
# simple filters first to shrink the data
print("Rows before early cleaning:", len(df))

# must-have fields for demand analysis
df = df.dropna(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
])

# validity filters
df = df[df["fare_amount"] > 0]
df = df[df["trip_distance"] > 0.2]
df = df[df["trip_distance"] <= 120]
df = df[df["PULocationID"].between(1, 263)]
df = df[df["DOLocationID"].between(1, 263)]

# passenger_count used as validation only
df = df[df["passenger_count"].notna()]
df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 5)]

print("Rows after early cleaning:", len(df))


# 4.create validation features
duration_seconds = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds()

df["trip_duration_min"] = duration_seconds / 60

del duration_seconds

# duration filters before speed calculation
df = df[(df["trip_duration_min"] > 1) & (df["trip_duration_min"] <= 300)]

# speed
df["speed_mph"] = df["trip_distance"] / (df["trip_duration_min"] / 60)

# speed filter
df = df[df["speed_mph"] <= 120]

print("Rows after duration/speed cleaning:", len(df))


# 5.duplicate removal
before_dupes = len(df)
df = df.drop_duplicates()
after_dupes = len(df)

print("Duplicates removed:", before_dupes - after_dupes)
print("Rows after duplicate removal:", len(df))

# 6.post-cleaning checks
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_after"))
display(df["passenger_count"].value_counts(dropna=False).sort_index().to_frame("count_after"))
display(df["trip_distance"].describe().to_frame("trip_distance_after"))
display(df["fare_amount"].describe().to_frame("fare_amount_after"))
display(df["trip_duration_min"].describe().to_frame("trip_duration_min_after"))
display(df["speed_mph"].describe().to_frame("speed_mph_after"))



# cleaned yellow taxi datasets for 21-25
YEARS = [2021, 2022, 2023, 2024, 2025]
MONTHS = range(1, 13)

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount"
]


def clean_trips(df):
    # all from above
    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")


    df = df.dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID"
    ])

    df = df[df["fare_amount"] > 0]
    df = df[df["trip_distance"].between(0.2, 120)]
    df = df[df["PULocationID"].between(1, 263)]
    df = df[df["DOLocationID"].between(1, 263)]

    df = df[df["passenger_count"].between(1, 5)]

    duration = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    df["trip_duration_min"] = duration

    df = df[df["trip_duration_min"].between(1, 300)]


    df["speed_mph"] = df["trip_distance"] / (df["trip_duration_min"] / 60)

    df = df[df["speed_mph"] <= 120]

    df = df.drop_duplicates()

    return df


for year in YEARS:

    print(f"\nProcessing {year}")

    monthly_frames = []

    #per month basis to not exhaust RAM as very large dataset 
    for month in MONTHS:

        url = f"{BASE_URL}/yellow_tripdata_{year}-{month:02d}.parquet"

        try:

            df = pd.read_parquet(url, columns=COLUMNS)

            df = clean_trips(df)

            monthly_frames.append(df)

            print(f"{year}-{month:02d} loaded:", len(df))

        except Exception as e:

            print(f"{year}-{month:02d} skipped:", e)

    # combine full year
    df_year = pd.concat(monthly_frames, ignore_index=True)

    print("Total trips after cleaning:", len(df_year))

    # hourly demand
    df_year["pickup_hour"] = df_year["tpep_pickup_datetime"].dt.floor("h")

    hourly_base = df_year[["PULocationID", "pickup_hour"]].copy()

    hourly_base["PULocationID"] = hourly_base["PULocationID"].astype("int16")

    observed = (
        hourly_base.value_counts(sort=False)
        .rename("trip_count")
        .reset_index()
    )

    # full hourly grid
    full_hours = pd.date_range(
        start=f"{year}-01-01 00:00:00",
        end=f"{year}-12-31 23:00:00",
        freq="h"
    )

    zones = pd.DataFrame(
        {"PULocationID": range(1, 264)},
        dtype="int16"
    )

    full_grid = (
        zones.assign(key=1)
        .merge(
            pd.DataFrame({"pickup_hour": full_hours}).assign(key=1),
            on="key"
        )
        .drop("key", axis=1)
    )

    # merge demand
    yearly_panel = full_grid.merge(
        observed,
        on=["PULocationID", "pickup_hour"],
        how="left"
    )

    yearly_panel["trip_count"] = yearly_panel["trip_count"].fillna(0).astype("int32")

    # engineer time features
    yearly_panel["date"] = yearly_panel["pickup_hour"].dt.normalize()
    yearly_panel["year"] = yearly_panel["pickup_hour"].dt.year
    yearly_panel["month"] = yearly_panel["pickup_hour"].dt.month
    yearly_panel["day"] = yearly_panel["pickup_hour"].dt.day
    yearly_panel["hour"] = yearly_panel["pickup_hour"].dt.hour

    yearly_panel["day_of_week"] = yearly_panel["pickup_hour"].dt.dayofweek

    yearly_panel["is_weekend"] = yearly_panel["day_of_week"] >= 5

    yearly_panel["is_rush_hour"] = yearly_panel["hour"].isin([7,8,9,17,18,19])

    yearly_panel = yearly_panel.sort_values(
        ["PULocationID","pickup_hour"]
    ).reset_index(drop=True)

    print("Final panel shape:", yearly_panel.shape)

    yearly_panel.to_parquet(
        f"yellow_hourly_demand_full_{year}.parquet",
        index=False
    )

    print(f"{year} saved")
